# Module 0: 環境設定與串流測試

## 學習目標
- 認識 Google Colab 與 GPU 加速
- 學會從全球電視串流擷取即時影像
- 基礎影像處理操作（灰階、邊緣偵測、模糊）

## 事前準備
1. 執行選單 **Runtime → Change runtime type → T4 GPU**
2. 從上到下依序執行每個 Cell（按 Shift+Enter）

---
## Step 1: 安裝必要套件

In [ ]:
# 安裝 AI 視覺辨識所需的套件（約 30 秒）
!pip install ultralytics opencv-python-headless pillow easyocr -q
print('所有套件安裝完成！')

## Step 2: 確認 GPU 可用

In [ ]:
# 確認 Colab 有分配到 GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU 已就緒: {gpu_name}')
    print(f'GPU 記憶體: {gpu_mem:.1f} GB')
else:
    print('未偵測到 GPU！')
    print('請到選單 Runtime → Change runtime type → 選擇 T4 GPU')

## Step 3: 載入工具函式

以下是本次工作坊會用到的顯示工具，**不需要修改，直接執行即可**。

In [ ]:
# === 工具函式（直接執行，不需修改）===
import cv2
import numpy as np
from PIL import Image
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import time

# 設定 matplotlib 中文顯示
plt.rcParams['font.size'] = 12

def show_frame(frame, title='', size=(640, 360)):
    """在 Colab 中顯示一幀影像"""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb).resize(size)
    if title:
        print(title)
    display(img)

def show_compare(images, titles, size=(320, 240)):
    """並排比較多張影像"""
    fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 4))
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def grab_frame(url, timeout=10):
    import subprocess
    cap = cv2.VideoCapture(url)
    cap.set(cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, timeout * 1000)
    ret, frame = cap.read()
    cap.release()
    if ret:
        return frame
    # 備援：台灣國道即時影像（穩定、無金鑰）——IPTV 串流常失效，這條保證有畫面
    print('串流連線失敗，改用台灣國道即時影像作為備援')
    data = subprocess.run(['curl', '-s', '--max-time', '8',
        'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=10001'],
        capture_output=True).stdout
    s = data.find(b'\xff\xd8'); e = data.find(b'\xff\xd9', s)
    if s >= 0 and e >= 0:
        return cv2.imdecode(np.frombuffer(data[s:e + 2], np.uint8), cv2.IMREAD_COLOR)
    raise ConnectionError('串流與備援皆失敗，請稍後再試')

print('工具函式載入完成！')

---
## Step 4: 串流來源清單

以下是來自全球合法公開電視串流（來源：[iptv-org](https://github.com/iptv-org/iptv)）。

每個場景都有多個備用串流，如果第一個連不上，試下一個即可。

In [ ]:
# === 全球即時串流來源 ===
# 每個場景都準備了多個備用來源，連不上就換下一個

STREAMS = {
    # --- 場景 1: 新聞台（人物 + 文字）---
    'Al Jazeera 半島電視台': 'https://live-hls-apps-aje-fa.getaj.net/AJE/index.m3u8',
    'DW 德國之聲': 'https://dwamdstream102.akamaized.net/hls/live/2015525/dwstream102/master.m3u8',
    'France 24 法國': 'https://amg00106-france24-france24-samsunguk-qvpp8.amagi.tv/playlist/amg00106-france24-france24-samsunguk/playlist.m3u8',
    'NHK World 日本': 'https://nhkwlive-xjp.akamaized.net/hls/live/2003458/nhkwlive-xjp/index_1080p.m3u8',
    'CGTN 中國': 'https://amg00405-rakutentv-cgtn-rakuten-i9tar.amagi.tv/master.m3u8',
    'ABC News 美國': 'https://abc-news-dmd-streams-1.akamaized.net/out/v1/701126012d044971b3fa89406a440133/index.m3u8',
    'Euronews 歐洲新聞': 'https://d35j504z0x2vu2.cloudfront.net/v1/master/0bc8e8376bd8417a1b6761138aa41c26c7309312/euronews/euronews-en.m3u8',

    # --- 場景 2: 購物台（各種商品/物件）---
    'QVC 購物台': 'https://qvc-amd-live.akamaized.net/hls/live/2034113/lsqvc2us/master.m3u8',
    'HSN 家庭購物': 'https://a-cdn.klowdtv.com/live2/hsn_720p/playlist.m3u8',
    'QVC UK 英國': 'https://qvcuk-live.akamaized.net/hls/live/2097112/qvc/master.m3u8',
    'Shop Channel 日本': 'https://stream3.shopch.jp/HLS/master.m3u8',
    'HSE 德國': 'https://hse24.akamaized.net/hls/live/2006663/hse24/playlist.m3u8',

    # --- 場景 3: 野生動物 / 自然 ---
    'WildEarth 野生地球': 'https://dqga3jatxofgx.cloudfront.net/WildEarth.m3u8',
    'Tanzania Safari 坦尚尼亞': 'https://stream-134630.castr.net/5fe35eae8c53540cab83659a/live_31dabe40323511f08b8efff0016f3b67/index.m3u8',
    'BBC Earth 亞洲': 'https://cdn4.skygo.mn/live/disk1/BBC_earth/HLSv3-FTA/BBC_earth.m3u8',
    'Adventure Earth': 'https://a57e9c69976649b582a8d7e604c00e69a.mediatailor.us-east-1.amazonaws.com/v1/master/44f73ba4d03e9607dcd9bebdcb8494d86964f1d8/RlaxxTV-eu_AdventureEarth/playlist.m3u8',

    # --- 場景 4: 運動（快速移動物體）---
    'Red Bull TV': 'https://3ea22335.wurl.com/master/f36d25e7e52f1ba8d7e56eb859c636563214f541/UmFrdXRlblRWLWdiX1JlZEJ1bGxUVl9ITFM/playlist.m3u8',
    'ESPN The Ocho': 'https://d3b6q2ou5kp8ke.cloudfront.net/ESPNTheOcho.m3u8',
    'beIN SPORTS': 'https://d35j504z0x2vu2.cloudfront.net/v1/master/0bc8e8376bd8417a1b6761138aa41c26c7309312/bein-sports-xtra/playlist.m3u8',

    # --- 場景 5: 旅遊 / 街景 ---
    '4K Travel TV': 'https://d35j504z0x2vu2.cloudfront.net/v1/master/0bc8e8376bd8417a1b6761138aa41c26c7309312/4k-travel-tv/manifest.m3u8',
    'DroneTV 空拍': 'https://d35j504z0x2vu2.cloudfront.net/v1/master/0bc8e8376bd8417a1b6761138aa41c26c7309312/dronetv/playlist.m3u8',
    'Beach TV Florida': 'http://media4.tripsmarter.com:1935/LiveTV/DTVHD/playlist.m3u8',
    'Schladming 奧地利山景': 'https://m317.video-stream-hosting.de/gzSoftware-live/_definst_/smil:livestream.smil/playlist.m3u8',
}

print(f'已載入 {len(STREAMS)} 個串流來源！')
print()
for name in STREAMS:
    print(f'  - {name}')

## Step 5: 抓取你的第一個即時畫面！

從上面的清單選一個頻道名稱，填入下方程式碼中。

In [ ]:
# === 動手做！===
# 把下面的頻道名稱換成你想看的
channel = 'Al Jazeera 半島電視台'

# 抓取一幀
frame = grab_frame(STREAMS[channel])
show_frame(frame, title=f'即時畫面: {channel}')

### 小練習：試試不同的頻道

把 `channel = '...'` 改成其他頻道名稱，重新執行。
如果某個頻道連不上，代表該串流目前離線，換一個就好。

---
## Step 6: 基礎影像處理

電腦看到的影像其實就是一堆數字。我們來看看常見的影像處理操作。

In [ ]:
# === 影像的本質 ===
print(f'影像形狀: {frame.shape}')
print(f'  → 高度: {frame.shape[0]} 像素')
print(f'  → 寬度: {frame.shape[1]} 像素')
print(f'  → 色彩通道: {frame.shape[2]} (藍、綠、紅)')
print(f'  → 總共 {frame.shape[0] * frame.shape[1]:,} 個像素')
print(f'  → 每個像素的值: 0（黑）~ 255（白）')
print()
print(f'左上角那個像素的顏色值: {frame[0, 0]}  (藍, 綠, 紅)')

In [ ]:
# === 灰階轉換 ===
# 把彩色變成黑白，很多演算法的第一步
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

# === 邊緣偵測 ===
# Canny 演算法：找出影像中物體的輪廓
edges = cv2.Canny(gray, 50, 150)

# === 模糊處理 ===
# 高斯模糊：讓影像變柔和，常用來去雜訊
blurred = cv2.GaussianBlur(frame, (21, 21), 0)

# 並排比較
show_compare(
    [frame, gray, edges, blurred],
    ['Original', 'Grayscale', 'Edge Detection', 'Gaussian Blur']
)

In [ ]:
# === 裁剪影像 ===
# 只取畫面中間的區域
h, w = frame.shape[:2]
cropped = frame[h//4 : 3*h//4, w//4 : 3*w//4]  # 中間 50% 區域

show_compare(
    [frame, cropped],
    ['Original', 'Cropped (Center 50%)']
)

---
## Step 7: 連續擷取多幀

即時串流是連續不斷的畫面。我們來擷取多幀，觀察畫面的變化。

In [ ]:
# === 從串流連續擷取 5 幀 ===
channel = 'DW 德國之聲'  # 可換成其他頻道
url = STREAMS[channel]

cap = cv2.VideoCapture(url)
frames = []

print(f'正在從 {channel} 擷取 5 幀...')
for i in range(5):
    ret, f = cap.read()
    if ret:
        frames.append(f)
        print(f'  第 {i+1} 幀 OK')
    time.sleep(1)  # 每秒抓一幀

cap.release()
print(f'擷取完成！共 {len(frames)} 幀')

In [ ]:
# === 顯示這 5 幀 ===
if frames:
    show_compare(
        frames[:5],
        [f'Frame {i+1}' for i in range(len(frames[:5]))]
    )

---
## Module 0 完成！

### 你學會了：
- Colab GPU 環境設定
- 從全球電視串流擷取即時影像
- 影像的本質（像素矩陣）
- 基礎影像處理：灰階、邊緣偵測、模糊、裁剪

### 接下來：
→ 開啟 **01_detection.ipynb**，讓 AI 來辨識畫面中的所有物件！